# Milestone 4 — Graded Questions: Formulating MCQ Task & Fine-Tuning

**Topics Covered:**
- Multiple-choice data formatting
- Tokenization for `AutoModelForMultipleChoice`
- Model outputs (logits & loss)
- LoRA (Low-Rank Adaptation) for efficient fine-tuning
- HuggingFace Trainer pipeline
- Inference & softmax probabilities

In [1]:
!pip install peft accelerate -q

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset
import warnings
warnings.filterwarnings('ignore')

train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
option_cols = ['A', 'B', 'C', 'D', 'E']
label_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}

print(f"Train shape: {train_df.shape}")
train_df.head()

Train shape: (2000, 8)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


## Q1 — Label Encoding

**Convert the `answer` column to numeric labels using A=0, B=1, C=2, D=3, E=4. What is the encoded numeric label for the row at index 150?**

In [2]:
train_df['label'] = train_df['answer'].map(label_map)

answer_q1 = train_df.loc[150, 'label']

print(f"Answer at index 150: '{train_df.loc[150, 'answer']}' → {answer_q1}")
print(f"\nAnswer: {answer_q1}")

Answer at index 150: 'C' → 2

Answer: 2


## Q2 — Prompt-Option Formatting

**For row index 0, create the Option B input using exactly: `str(prompt) + " [SEP] " + str(option_B)`. What is the exact character length?**

This is how we format inputs for multiple-choice models — each option is paired with the prompt as a separate sequence.

In [3]:
row_0 = train_df.iloc[0]

formatted_b = str(row_0['prompt']) + " [SEP] " + str(row_0['B'])

print(f"Formatted string:\n{formatted_b}\n")

answer_q2 = len(formatted_b)
print(f"Answer (character length): {answer_q2}")

Formatted string:
Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options. [SEP] Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.

Answer (character length): 407


## Q3 — Single-Row MCQ Tokenization

**Using `bert-base-uncased`, tokenize the 5 formatted inputs for row index 0 with `padding='max_length'`, `truncation=True`, `max_length=128`, `return_tensors='pt'`. After reshaping to `[1, 5, 128]`, what is the value of the second dimension?**

Multiple-choice models expect shape `[batch_size, num_choices, seq_length]`.

In [4]:
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

row_0 = train_df.iloc[0]

# Create 5 formatted inputs
formatted_inputs = []
for col in option_cols:
    text = str(row_0['prompt']) + " [SEP] " + str(row_0[col])
    formatted_inputs.append(text)

# Tokenize all 5
tokenized = tokenizer(
    formatted_inputs,
    padding='max_length',
    truncation=True,
    max_length=128,
    return_tensors='pt'
)

print(f"Tokenized input_ids shape (before reshape): {tokenized['input_ids'].shape}")

# Reshape for multiple-choice model: [1, 5, 128]
input_ids = tokenized['input_ids'].unsqueeze(0)  # add batch dimension
print(f"After reshape: {input_ids.shape}")

answer_q3 = input_ids.shape[1]
print(f"\nAnswer (second dimension): {answer_q3}")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenized input_ids shape (before reshape): torch.Size([5, 128])
After reshape: torch.Size([1, 5, 128])

Answer (second dimension): 5


## Q4 — Batch MCQ Tokenization

**Tokenize the first 16 rows as multiple-choice examples. Each row has 5 choices, each tokenized to length 128. Final shape: `[16, 5, 128]`. How many total token positions are in this tensor?**

Total positions = batch_size × num_choices × seq_length

In [5]:
batch_input_ids = []

for idx in range(16):
    row = train_df.iloc[idx]
    formatted = [str(row['prompt']) + " [SEP] " + str(row[col]) for col in option_cols]

    tok = tokenizer(
        formatted,
        padding='max_length',
        truncation=True,
        max_length=128,
        return_tensors='pt'
    )
    batch_input_ids.append(tok['input_ids'])

# Stack into batch: [16, 5, 128]
batch_tensor = torch.stack(batch_input_ids)
print(f"Batch tensor shape: {batch_tensor.shape}")

answer_q4 = batch_tensor.shape[0] * batch_tensor.shape[1] * batch_tensor.shape[2]
print(f"\nAnswer (total token positions): {answer_q4}")

Batch tensor shape: torch.Size([16, 5, 128])

Answer (total token positions): 10240


## Q5 — Multiple-Choice Logits

**Load `bert-base-uncased` using `AutoModelForMultipleChoice`. Tokenize row index 0 as 5 choices and pass through the model. The output logits have shape `[1, 5]`. How many logits are produced for one question?**

The model produces one logit per option — each logit represents how "correct" the model thinks that option is.

In [6]:
# Load multiple-choice model
mc_model = AutoModelForMultipleChoice.from_pretrained('bert-base-uncased')
mc_model.eval()

# Tokenize row 0 as 5 choices
row_0 = train_df.iloc[0]
formatted = [str(row_0['prompt']) + " [SEP] " + str(row_0[col]) for col in option_cols]

tok = tokenizer(
    formatted,
    padding='max_length',
    truncation=True,
    max_length=128,
    return_tensors='pt'
)

# Reshape for multiple-choice: add batch dimension [1, 5, 128]
mc_input = {k: v.unsqueeze(0) for k, v in tok.items()}

# Forward pass
with torch.no_grad():
    outputs = mc_model(**mc_input)

print(f"Logits shape: {outputs.logits.shape}")
print(f"Logits: {outputs.logits}")

answer_q5 = outputs.logits.shape[1]
print(f"\nAnswer (number of logits): {answer_q5}")

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Logits shape: torch.Size([1, 5])
Logits: tensor([[0.2307, 0.1954, 0.2053, 0.2081, 0.2223]])

Answer (number of logits): 5


## Q6 — Supervised Loss Tensor

**Pass the tokenized 5-choice input for row 0 along with the correct label. The model returns a scalar loss tensor. How many dimensions does it have?**

A scalar tensor has 0 dimensions (`tensor.dim() == 0`). The loss is computed using cross-entropy between the 5 logits and the correct label index.

In [7]:
# Get the correct label for row 0
label = torch.tensor([label_map[row_0['answer']]])

# Forward pass with labels — model computes loss automatically
with torch.no_grad():
    outputs_with_loss = mc_model(**mc_input, labels=label)

print(f"Loss value: {outputs_with_loss.loss.item():.4f}")
print(f"Loss tensor: {outputs_with_loss.loss}")
print(f"Loss dimensions: {outputs_with_loss.loss.dim()}")

answer_q6 = outputs_with_loss.loss.dim()
print(f"\nAnswer (dimensions): {answer_q6}")

Loss value: 1.6265
Loss tensor: 1.6265321969985962
Loss dimensions: 0

Answer (dimensions): 0


## Q7 — LoRA Trainable Parameters

**Apply LoRA to the `bert-base-uncased` multiple-choice model with:**
- `r=8`, `lora_alpha=16`
- `target_modules=["query", "value"]`
- `lora_dropout=0.1`, `bias="none"`
- `task_type=TaskType.SEQ_CLS`

**Count trainable parameters. How many are trainable?**

LoRA freezes the original model and adds small trainable adapter matrices to the specified attention layers.

In [8]:
!pip install torchao --upgrade -q
!pip install peft accelerate -q
# Load fresh model for LoRA
lora_base_model = AutoModelForMultipleChoice.from_pretrained('bert-base-uncased')

# Configure LoRA
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS,
)

# Apply LoRA
lora_model = get_peft_model(lora_base_model, lora_config)

# Count trainable parameters
trainable_params = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in lora_model.parameters())

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Trainable %:          {100 * trainable_params / total_params:.2f}%")

lora_model.print_trainable_parameters()

answer_q7 = trainable_params
print(f"\nAnswer (trainable parameters): {answer_q7}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 76.1 MB/s eta 0:00:00


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Skipping import of cpp extensions due to inco

Total parameters:     109,778,690
Trainable parameters: 295,681
Trainable %:          0.27%
trainable params: 295,681 || all params: 109,778,690 || trainable%: 0.2693

Answer (trainable parameters): 295681


## Q8 — HuggingFace Dataset Preparation

**Create a HuggingFace `Dataset` from the first 100 rows of train.csv. Each row should have `input_ids` with shape `[5, 128]`, `attention_mask` with shape `[5, 128]`, and `labels` as the encoded answer. How many tokenized choices are stored in `input_ids`?**

In [9]:
def tokenize_mcq(examples):
    """Tokenize a batch of MCQ examples for multiple-choice format."""
    all_input_ids = []
    all_attention_mask = []

    # Process each example in the batch
    for i in range(len(examples['prompt'])):
        formatted = [
            str(examples['prompt'][i]) + " [SEP] " + str(examples[col][i])
            for col in option_cols
        ]

        tok = tokenizer(
            formatted,
            padding='max_length',
            truncation=True,
            max_length=128,
            return_tensors='pt'
        )

        all_input_ids.append(tok['input_ids'])        # shape: [5, 128]
        all_attention_mask.append(tok['attention_mask'])  # shape: [5, 128]

    return {
        'input_ids': all_input_ids,
        'attention_mask': all_attention_mask,
        'labels': [label_map[a] for a in examples['answer']],
    }

# Create dataset from first 100 rows
first_100 = train_df.head(100)
hf_dataset = Dataset.from_pandas(first_100[['prompt'] + option_cols + ['answer']])

# Tokenize
hf_dataset = hf_dataset.map(tokenize_mcq, batched=True, batch_size=100)
hf_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

# Check shapes
first_item = hf_dataset[0]
print(f"input_ids shape:      {first_item['input_ids'].shape}")
print(f"attention_mask shape: {first_item['attention_mask'].shape}")
print(f"label:                {first_item['labels']}")

answer_q8 = first_item['input_ids'].shape[0]
print(f"\nAnswer (number of choices in input_ids): {answer_q8}")

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

input_ids shape:      torch.Size([5, 128])
attention_mask shape: torch.Size([5, 128])
label:                1

Answer (number of choices in input_ids): 5


## Q9 — Tiny LoRA Fine-Tuning

**Fine-tune the LoRA multiple-choice model on the first 32 rows using HuggingFace Trainer with:**
- `max_length = 64`
- `per_device_train_batch_size = 4`
- `gradient_accumulation_steps = 1`
- `max_steps = 4`

**What is the final `global_step` reported by the Trainer?**

In [10]:
# Prepare dataset with first 32 rows and max_length=64
def tokenize_mcq_64(examples):
    all_input_ids = []
    all_attention_mask = []

    for i in range(len(examples['prompt'])):
        formatted = [
            str(examples['prompt'][i]) + " [SEP] " + str(examples[col][i])
            for col in option_cols
        ]

        tok = tokenizer(
            formatted,
            padding='max_length',
            truncation=True,
            max_length=64,
            return_tensors='pt'
        )

        all_input_ids.append(tok['input_ids'])
        all_attention_mask.append(tok['attention_mask'])

    return {
        'input_ids': all_input_ids,
        'attention_mask': all_attention_mask,
        'labels': [label_map[a] for a in examples['answer']],
    }

# Create dataset from first 32 rows
first_32 = train_df.head(32)
train_ds = Dataset.from_pandas(first_32[['prompt'] + option_cols + ['answer']])
train_ds = train_ds.map(tokenize_mcq_64, batched=True, batch_size=32)
train_ds.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

print(f"Training dataset: {len(train_ds)} samples")

# Load fresh LoRA model for training
ft_base = AutoModelForMultipleChoice.from_pretrained('bert-base-uncased')
ft_model = get_peft_model(ft_base, lora_config)

# Training arguments
training_args = TrainingArguments(
    output_dir='./tiny_ft',
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    max_steps=4,
    logging_steps=1,
    report_to='none',
    save_strategy='no',
    remove_unused_columns=False,
)

# Custom data collator
class MCQCollator:
    def __call__(self, features):
        batch = {}
        batch['input_ids'] = torch.stack([f['input_ids'] for f in features])
        batch['attention_mask'] = torch.stack([f['attention_mask'] for f in features])
        batch['labels'] = torch.tensor([f['labels'] for f in features])
        return batch

# Train
trainer = Trainer(
    model=ft_model,
    args=training_args,
    train_dataset=train_ds,
    data_collator=MCQCollator(),
)

result = trainer.train()

answer_q9 = trainer.state.global_step
print(f"\nAnswer (final global_step): {answer_q9}")

Map:   0%|          | 0/32 [00:00<?, ? examples/s]

Training dataset: 32 samples


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
1,3.296396
2,3.259986
3,3.210134
4,3.180639



Answer (final global_step): 4


## Q10 — Probability Assigned to Option E After Fine-Tuning

**Using the fine-tuned LoRA model from Q9, run inference on row index 0 and apply softmax to the logits. What is the probability assigned to Option E? (Round to 4 decimal places)**

Option E is at index 4 (A=0, B=1, C=2, D=3, E=4). We apply softmax to convert raw logits into probabilities that sum to 1.0.

In [11]:
# Prepare row 0 input with max_length=64 (same as training)
row_0 = train_df.iloc[0]
formatted = [str(row_0['prompt']) + " [SEP] " + str(row_0[col]) for col in option_cols]

tok = tokenizer(
    formatted,
    padding='max_length',
    truncation=True,
    max_length=64,
    return_tensors='pt'
)

# Move inputs to same device as model
device = next(ft_model.parameters()).device
mc_input = {k: v.unsqueeze(0).to(device) for k, v in tok.items()}

# Run inference
ft_model.eval()
with torch.no_grad():
    outputs = ft_model(**mc_input)

logits = outputs.logits
print(f"Raw logits: {logits}")

probs = F.softmax(logits, dim=-1)
print(f"Probabilities: {probs}")

for i, col in enumerate(option_cols):
    print(f"  Option {col}: {probs[0][i].item():.4f}")

answer_q10 = round(probs[0][4].item(), 4)
print(f"\nAnswer (Option E probability): {answer_q10}")

Raw logits: tensor([[0.2951, 0.2082, 0.2654, 0.2954, 0.3160]], device='cuda:0')
Probabilities: tensor([[0.2037, 0.1867, 0.1977, 0.2038, 0.2080]], device='cuda:0')
  Option A: 0.2037
  Option B: 0.1867
  Option C: 0.1977
  Option D: 0.2038
  Option E: 0.2080

Answer (Option E probability): 0.208
